## COMP 341: Practical Machine Learning
## Homework Assignment 7: Is a Picture Worth 1000 Words?
### Due: Friday, December 2 at 11:59pm on Gradescope

In Homework 4, we used regression-based methods to predict the list prices of homes in the Houston area. Now, we will use both classic ML and deep learning techniques to predict a home's list price based on some of the same features we have seen before (square footage, # of bathrooms, etc.) but also the first picture on the house listing. As before, the data has been obtained from recently listed homes on [Redfin](https://www.redfin.com).

As always, fill in missing code following `# TODO:` comments or `####### YOUR CODE HERE ########` blocks and be sure to answer the short answer questions marked with `[WRITE YOUR ANSWER HERE]` in the text.

All code in this notebook will be run sequentially so make sure things work in order! Be sure to also use good coding practices (e.g., logical variable names, comments as needed, etc), and make plots that are clear and legible.

For this assignment, there will be **15 points** allocated for general coding points:
* **10 points** for coding style
* **5 points** for code flow (accurate results when everything is run sequentially)

### Part 0: Setup
First, we need to import some libraries that are necessary to complete the assignment.

In [ ]:
import pandas as pd
import numpy as np

from PIL import Image, ImageOps
import os

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim

from collections import defaultdict

Add additional modules/libraries to import here (rather than wherever you first use them below):

In [ ]:
# additional modules/libraries to import

Packages such as [TorchMetrics](https://torchmetrics.readthedocs.io/en/stable/) provide more options for evaluation metrics than what PyTorch natively provides. Here, we install the package and load their implementation of MSE, since it natively supports RMSE (which we will use in Part 2 instead of implementing it from scratch).

In [ ]:
!pip install torchmetrics

In [ ]:
from torchmetrics import MeanSquaredError

As in Homework 6, we would want to take advantage of any GPU resources to speed up our training. As with before, click on the RAM / Disk monitoring bars towards the upper right corner of your Colab instance. Then in the menu that appears, click "Change runtime type" and select "GPU" under "Hardware accelerator." Let's check if the GPU is indeed detected. Note that whis code snippet will by default assume we are using CPU resources unless it can detect GPUs available.

In [ ]:
device = "cpu"
if (torch.cuda.is_available()):
  device = "cuda"
print("device: " + device)

If `device` is now `cuda`, that means GPUs have been successfully enabled. The `device` string simply will be used in a few key parts of the code to signal to PyTorch whether or not to move the model / weights etc to the GPU, if available.Using GPUs for this assignment is optional as before, but because the dataset and images are larger, runtimes can be reduced greatly with GPUs.

As in previous assignments, we will use Google Drive to access the dataset. However, there are some slight modifications once more because your `DataLoader` will be frequently accessing the raw image files, which can be very slow if it is having to access the files on Google Drive constantly.

By now, you will probably already created your local `comp341` directory:
1. Go to 'My Drive' in your own Google Drive
2. Make a new folder named `comp341`

Now, we will copy a shortcut to a zipped data file with all of the images and metadata for the houses:
3. From the [Google Drive link](https://drive.google.com/drive/folders/1SUaLUMDZ8A8EXHx_7GZeUYMe_0yPEbA-?usp=sharing), you can right click the `comp341-hw7.zip` file, and select `Add shortcut to Drive`, and add a link to the file to your `comp341` folder. This is a convenient alternative to having to download and re-upload the files to your own drive.

If you run into trouble with accessing the files from the shortcut, then:
4. Download the `comp341-hw7.zip` file and click `New -> File Upload` to upload the downloaded file from your computer.

Now, we will mount your Google Drive as usual, but we will have an additional step of copying over the zip file to your local colab instance so that files can be accessed quickly.

In [ ]:
# note that this command will trigger a request from google to allow colab
# to access your files: you will need to accept the terms in order to access
# the files this way
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# if you followed the instructions above exactly, your zipped data file should
# be located at the file path below; if your files are in a different directory
# on your Google Drive, you will need to change the path below accordingly
ZIPPATH = '/content/drive/My Drive/comp341/comp341-hw7.zip'

In [ ]:
!cp "{ZIPPATH}" .
!unzip -q "comp341-hw7.zip"
!rm "comp341-hw7.zip"

In your local colab instance, you should now have a `house_imgs/` directory with many images of homes (includes images from both the training and test sets), as well as two csv files: `home_data_train.csv` and `home_data_test.csv`.

### Part 1: Exploring the Home Images
We have explored various tabular data extensively, especially in the context of dimensionality reduction when we have many features. One way to think about images is to consider each individual pixel (and channel) as an individual feature. Even with relatively small images like we have in our dataset, the dimensionality explodes pretty quickly, so let's explore if the dimensionality reduction methods we covered early on in the course can help us make sense of the data before we do any type of more sophisticated feature extraction.

In [ ]:
# we provide some simple code to read in each training image and flatten
# the pixel-based values to a tidy DataFrame, where each row is a house image
# and each column is a feature (the R/G/B value at an individual pixel location)

# get houseids for homes in the training dataset
home_train = pd.read_csv('home_data_train.csv')
img_ids = home_train['houseid'].astype(str).tolist()

img_vect = []
for idx in img_ids:
  infile = os.path.join("house_imgs", idx + ".jpg")
  file, ext = os.path.splitext(infile)
  with Image.open(infile) as im:
    img_vect.append(np.asarray(im).flatten())

pixel_df = pd.DataFrame(np.vstack(img_vect))

Here, we store the flattened pixel values for each image, but we may want to look at the original image that corresponds to these flattened vectors.

In [ ]:
# TODO: write a simple function that, given a single house id as input (aka one of the elements
# in img_ids), loads the image file from its location in LOCALDIR and displays it in your notebook directly
# Hint: this is essentially a simpler version of the display_data function we provided in
# Part 2 of the assignment, so you may be able to use some of the similar methods referenced
# there as well as the provided HouseImagesDataset code

In [ ]:
# TODO: test your function on one of the images in img_ids

Now, use a dimensionality reduction method that we covered in class that you think is appropriate for this problem.

In [ ]:
# TODO: calculate the 2D reduced dimensionality space and plot it (each image is a single point)
# Note: depending on the dimensionality reduction method chosen,
# this step can take a couple of minutes to complete

If we look closely at some of our house images, we can see that instead of providing a "real" picture of the house, there are also schematics / floorplans. Two such examples are the houses at `houseid` 4112 and 7758.


In [ ]:
# TODO: use your image display function to verify that these 2 houseids used drawings instead of real pictures of the house(s).

In [ ]:
# TODO: color the 2 points that correspond to these 2 houses in a new 2D reduced dimensionality plot

In [ ]:
# TODO: using the dimensionality reduction results and/or any algorithms we covered in class, estimate
# the number of houses in our training data that have a drawing as their primary image; correctness of this
# estimate will be considered for grading purposes, but with a generous window of error. Feel free to check
# your predictions using your image display function

### Part 2: Predicting List Price
Since we explored this task in Homework 4, we are already somewhat familiar with the tabular features, including some of the data cleaning / preprocessing steps that might be valuable, as well as which ones may be more or less valuable to keep.

Now that we also have some sense of what the house listing images are like based on Part 1, we will set up a regression framework that can use both the tabular features and image features. Along the way, we will also see how our predictions may change depending on what data we use.

We provide several helper functions for setting up the data and functionality to visualize individual examples, which can sometimes be helpful to get a sense of what the model is doing.

In [ ]:
# torch converts the 0-255 RGB values to 0-1 tensors, but it can
# also be beneficial to also standardize the values (or, as we
# see here, subtract the mean RGB values from the images)

# these transformations below help facilitate this
# inv_normalize is provided mainly for visualization sake, so that
# we can flip the standardization process to see the image in its
# original colors

house_mean = [0.5230, 0.5416, 0.4989]
# house_sd = [0.2271, 0.2162, 0.2640]
# only subtracting mean and not also dividing by standard deviation
# can actually sometimes work better, which is what we are doing here
house_sd = [1, 1, 1]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(house_mean, house_sd)
])

inv_normalize = transforms.Normalize(
   mean= [-m/s for m, s in zip(house_mean, house_sd)],
   std= [1/s for s in house_sd]
)


In [ ]:
# convenient function for displaying images
# by default, will reverse the standardization calculation so that we can
# see the images in a "normal" color scheme
def display_data(d, inv_norm=True):
  if isinstance(d['houseid'], list): # we can handle a list of houses
    batch_size = len(d['houseid'])
    for i in range(batch_size):
      if 'price' in d:
        print('price:', "${:,.0f}".format(d['price'][i]))

      if inv_norm:
       display(transforms.ToPILImage()(inv_normalize(d['image'][i])))
      else:
        display(transforms.ToPILImage()(d['image'][i]))
  else: # only an individual house to be displayed
    if 'price' in d:
      print('price:', "${:,.0f}".format(d['price']))

    if inv_norm:
      display(transforms.ToPILImage()(inv_normalize(d['image'])))
    else:
      display(transforms.ToPILImage()(d['image']))

As you may recall, data handling in PyTorch relies heavily on the `Dataset` class. In Homework 6, we used one of the pre-built `Datasets`, but most of the time, we need to set up our own. Here, we provide a scaffold `HouseImagesDataset` class.

You will need to fill in any cleaning / data preprocessing steps for the features (recall the best practices we covered in Homeworks 1-4 in terms of missing value handling, scaling, one-hot encoding) in `__init__`. We provide code for `__getitem__`, but you will also need to fill in the other method we typically define for a custom `Dataset`: `__len__`.

In [ ]:
class HouseImagesDataset(Dataset):
    def __init__(self, annot_file, image_dir, train=True):
        # the annotation file is tidy, aka each row is a unique observation in the dataset,
        # but it is not yet clean, which you will address in the TODO below
        df = pd.read_csv(annot_file)

        # TODO: cleaning / preprocessing of features in df

        # TODO: fill in this feature_cols list with the column names of
        # features you would like to use to predict list price (many of the columns
        # will likely be transformed from the original data in annot_file)
        self.feature_cols = []

        self.house_annot = df
        self.image_dir = image_dir
        self.train=train

    def __len__(self):
        # TODO: fill in this method (replacing pass) to return the length of the dataset

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        # we have provided code that will load and transform the indexed ("ith") image
        # as well as features specified earlier in self.feature_cols within the processed
        # pandas DataFrame
        img = Image.open(os.path.join(self.image_dir, self.house_annot.loc[idx, 'houseid'] + '.jpg'))
        img = transform(img)

        features = self.house_annot.loc[idx, self.feature_cols]
        features = features.tolist()
        features = torch.FloatTensor(features)

        # depending on whether the Dataset is in training mode, we will have the object data or not
        if self.train:
            item = {'image': img,
                    'id': self.house_annot.loc[idx, 'houseid'],
                    'features': features,
                    'object': torch.tensor(self.house_annot.loc[idx, 'list_price'], dtype=torch.float)}
        else:
            item = {'image': img,
                    'id': self.house_annot.loc[idx, 'houseid'],
                    'features': features}

        return item

In [ ]:
# TODO: initialize the house dataset using the training data you were provided and check the length of the dataset

In [ ]:
# TODO: check that the data loads properly by calling the provided display_data function with a specifically indexed
# item in your house dataset (e.g., house_dataset[3])

In [ ]:
# TODO: use the convenient torch.utils.data.random_split function to split your loaded dataset into training and
# validation portions, using 75% of the data for training and 25% of the data for validation

We explored how to set up a CNN with several convolutional layers and some fully connected layers in Homework 6 when we have image data. Now, we have *both* image and tabular features. One natural way we can handle this is to set up several convolutional layers to extract important image-related features, and in parallel, pass our tabular features that we indicated we want to use (`self.feature_cols` in our `HouseImagesDataset`) through a simple MLP, then simply **concatenate** the two sets of features together before passing them through a set of linear layers before reaching our final prediction.

Recall that with house price predictions, we want to end up with a single non-zero prediction for each house.

In [ ]:
class HybridHouseNN(nn.Module):
  def __init__(self):
    # TODO: set up some convolutional layers
    # it is your choice as to how many convolutional blocks, as well as
    # specifics within the blocks: activation function, pooling, etc

    # TODO: set up an MLP for the tabular features that you will be
    # inputting into this model

    # TODO: set up the final set of fully connected layers that
    # takes as input the concatenated set of flattened convolution features
    # together with the output of the MLP from the tabular features
    # to eventually output a single non-negative prediction

  def forward(self, ximg, xfeats):
    # TODO: write out the forward pass steps
    # note that forward now has 2 inputs because we are using both
    # images and non-image features separately at first, before
    # merging them together for the final set of predictions
    # Note: you may also need to adjust the shape of your final prediction
    # so that it plays nice with the loss function etc.


Before training our model, we want to also set up some additional models to what the differences might be if we use *only* images or *only* the tabular features for our predictions. Of course, if we set the models up differently with different hyperparameters, we really cannot have a truly equivalent comparison, but we will try to keep as many of the model blocks the same as possible.

In [ ]:
class HouseImageOnly(nn.Module):
  def __init__(self):
    # TODO: set up this model to only use images for predictions,
    # using the same convolutional blocks that you designed for
    # your HybridHouseNN model; also use a similar set of
    # fully connected layers (changing only the input dimension
    # since you will no longer be concantenating other features)

  def forward(self, ximg):
    # TODO: the forward pass will now only have the image as input


In [ ]:
class HouseFeatsOnly(nn.Module):
  def __init__(self):
    # TODO: set up this model to only use the tabular, non-image features
    # this model will simply be a MLP with those features as input
    # once again, try to keep the layers relatively comparable to what
    # you chose in HybridHouseNN, though of course the dimensionality going
    # into the last set of fully connected layers will likely change drastically
    # without image features

  def forward(self, xfeats):
    # TODO: the forward pass will now only have non-image features as input


As mentioned earlier, we will use RMSE for our loss function:

In [ ]:
loss_fn = MeanSquaredError(squared=False).to(device)

Now, let's fill in the details of the training and validation methods.

In [ ]:
def train(model, train_loader, opt, epoch, mode="both", verbose=False):
  # mode can be "both", "image", or "features", depending on if we are using
  # our HybridHouseNN, HouseImageOnly, or HouseFeatsOnly model
  # we will assume that the model passed to this function matches the mode,
  # and mode will affect whether the model uses image, features, or a combination
  # as input to get the predictions in the forward pass

  if verbose:
    print("starting epoch", epoch)
  train_loss = 0
  for i, batch in enumerate(train_loader):
    image, features, price = batch['image'].to(device), \
                             batch['features'].to(device), \
                             batch['price'].to(device)

    model.train(True)

    # TODO: fill in the code for each of the steps in the
    # training loop, remembering that we want to account for
    # different modes in the forward pass step

    model.train(False)
    if verbose and ((i % 20) == 0):
      print('training [epoch {}: {}/{} ({:.0f}%)] loss: {:.6f}'.format(
          epoch, i * len(image), len(train_loader.dataset),
          100. * i / len(train_loader), loss.item()))

  avg_tl = train_loss / (i+1)
  print('epoch {} avg training loss: {:.6f}'.format(epoch, avg_tl))
  return avg_tl

In [ ]:
def valid(model, valid_loader, mode="both"):
  # as in train, mode can be "both", "image", or "features", depending on if we are using
  # our HybridHouseNN, HouseImageOnly, or HouseFeatsOnly model
  # we will assume that the model passed to this function matches the mode,
  # and mode will affect whether the model uses image, features, or a combination
  # as input to get the predictions in the forward pass
  valid_loss = 0
  correct = 0
  with torch.no_grad():
    for i, batch in enumerate(valid_loader):
      image, features, price = batch['image'].to(device), \
                               batch['features'].to(device), \
                               batch['price'].to(device)

      # TODO: fill in code to calculate pred (the prediction), paying attention to
      # different usage of the model depending on the inputted mode variable

      # get loss
      valid_loss += loss_fn(pred, price).item()


  # get the loss for the epoch
  avg_vl = valid_loss / (i+1)
  print('avg validation loss: {:.6f}'.format(avg_vl))

  return avg_vl

In [ ]:
# TODO: initialize your training and validation DataLoaders, using a batch_size of 64
# and remembering to randomize the order the data points are presented to the model

In [ ]:
# we will use these simple dictionaries to keep track of the loss for our 3 different models
# note that depending on how complex your 3 models are, training may take some time, but should
# not take longer than 30 minutes or so
# with verbose on below, you should be seeing continual progress during training: if not, then
# double check that you are using GPUs and the image files locally within your colab instance
epoch_list = defaultdict(list)
train_loss = defaultdict(list)
valid_loss = defaultdict(list)

epochs = 30
modes = {'both': HybridHouseNN(), 'image': HouseImageOnly(), 'features': HouseFeatsOnly()}
for m in modes:
  model = modes[m].to(device)
  # TODO: initialize the optimizer (and associated hyperparameters like learning rate) of your choice
  opt = optim.#OPTIMIZER

  print("Current mode:", m)
  for e in range(1, epochs+1):
    epoch_list[m].append(e)
    train_loss[m].append(train(model, train_loader, opt, e, m, verbose=True))
    valid_loss[m].append(valid(model, valid_loader, m))

In [ ]:
# TODO: on a single plot, plot the training and validation losses for your 3 different models

**Short Answer Question:** Based on the results that you see in your plot, what do you think about using images, tabular features, or both for predicting list price? (Of course, the results and associated interpretations may change depending on how the models are set up, so describe your interpretations solely based on the plot that you generated above and not other potential possibilities.)

`[WRITE YOUR ANSWER HERE]`

### Part 3: Optimizing Performance
As before, we will be using [Kaggle](https://www.kaggle.com/t/c9759b93a1444f9e874b523524a3852c) with held out data to test the performance of our model.

For this homework, *you will be graded on your ability to pass perfomance benchmarks*. To receive full credit for this section of the assignment you will need to pass 3 benchmarks (baseline, easy, moderate) ***on both the public and private leaderboard***. Remember that since the private leaderboard is a part of your grade, you want to refrain from solely overfitting to the public leaderboard! Partial credit will be allotted depending on how many benchmarks are passed.


The top three leaders on the private leaderboard will recieve extra credit (if there are ties, everyone tied will receive the same number of points):
* 5 points for first place
* 3 points for second place
* 2 points for third place



The following Kaggle notes from the previous assignments still apply:
* You can use any team name (the name that will show up on the Kaggle leaderboard) as long as it is not inappropriate or offensive; however, in order to receive credit, you **must** specify your `team name` in your notebook here. If you do not, there is no way for us to assign you credit!
* Kaggle lists the close date as several days after the homework's due date. This is because Kaggle does not support late submissions. The homework and your submission on Kaggle are due by the due date listed here, but you may use late days and turn it in late (i.e., if you submit Kaggle predictions after the due date, it will automatically count towards your late days even if you have turned in your notebook already).
* This portion of the assignment **must** be completed independently. You cannot share prediction code or predictions with each other. In fact, you must put the exact code you use for your final predictions below. Violations will result in point deductions.
* Related, you cannot modify your prediction files manually. Violations will result in point deductions.
* **Unlike previous assignments**, you are free to use *any* regression models you would like for this kaggle competition!

### Some helpful tips:

* You are not required to use all of training data for this challenge: strategically applying filters and/or transformations (on observations or features, or both) can be helpful
* Remember that you are free to use *any* regression model you would like, regardless of whether we discussed it in class---the world is your oyster!
* As always, you want to check if your model is robust to different validation sets so that you are not overfitting to the public leaderboard.
* The extra credit in Part 4 can be synergistic in terms of guiding model tuning if you are using deep learning models.



**Kaggle team name:** `[fill in here]`

Read in the test dataset (`home_data_test.csv` and load associated images if you are using them).

In [ ]:
# TODO: put all code needed (including preprocessing steps) to make your
# final kaggle submission; note that this code must match the predictions
# that you provide on kaggle

You can see details about the file format for submission on kaggle (`sample_submission.csv`, essentially a 2 column file with `houseid`, the unique identifier in your test set, and `price`, your predictions). To make things easier, we provide here some sample code that you can modify to make your own submission file if your predictions were in a variable called `y_pred_kagg`. On Kaggle, the performance metric is RMSE as it was in Part 2 of this assignment.

In [ ]:
results = pd.Series(y_pred_kagg.flatten(), name="price")
results = pd.concat([test_bag['houseid'], results], axis=1)
results.to_csv('my_submission.csv', index=False)

Once you output your csv file, you need to download the file from colab to your local computer (you can click the file folder icon on the left panel to see the files in your workspace) and upload that file to the Kaggle site as your submission. Note that you can submit multiple times (up to 10 times a day)!

### Part 4: TensorBoard [Extra Credit: up to 5 points]

As we discussed in class, there are overwhelming numbers of hyperparameters in deep learning. One of the innovations of TensorFlow was helpful visualizations that can help guide the tuning of this hyperparameters, especially through [TensorBoard](https://www.tensorflow.org/tensorboard). Though this was originally developed as part of TensorFlow, TensorBoard is also available with [PyTorch](https://pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html)!

Here, set up TensorBoard for the hybrid (CNN + MLP) model you set up in Part 2 to log your training and validation loss and use it to help guide your choice of epochs and learning rate, as well as other hyperparameters you may be interested in!

To get extra credit, use the TensorBoard magic commands (%tensorboard) available through TensorFlow to set this up so that you can directly view the TensorBoard in your notebook. If implemented correctly, you will see that this can help give you a quicker sense of which hyperparameters might be better.